# N092 · 网络流、最小割与匹配归约

**目标：** 在小网络上实现流并以割容量验证最优性。

**先修：** N084, N087, N082。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 容量；流守恒；残量网络；反向边；Dinic层次图；最大流最小割。

## 从问题到算法

最大流同时受容量上界和中间点守恒约束。反向残量边表示撤销先前选择的能力，而非原图真的多了一条运输路线。Dinic先构建最短层次图，再在当前层次图中发送阻塞流。下面使用显式路径栈替代递归DFS；当前弧指针避免反复扫描已无用的边。

## 最小实现

**本章接口：** `Dinic.add_edge(u, v, capacity)`、`Dinic.max_flow(source, sink)`、`min_cut_reachable(source)`、`matching_via_flow(left_adj, right_size)`

In [1]:
from collections import deque

class Dinic:
    def __init__(self,n): self.adj=[[] for _ in range(n)]; self.original=[]
    def add_edge(self,u,v,capacity):
        if capacity<0: raise ValueError('nonnegative capacity required')
        i,j=len(self.adj[u]),len(self.adj[v])
        self.adj[u].append([v,j+(u==v),capacity])
        self.adj[v].append([u,i,0]); self.original.append((u,i,capacity))
    def _levels(self,s):
        level=[-1]*len(self.adj); level[s]=0; q=deque([s])
        while q:
            u=q.popleft()
            for v,rev,cap in self.adj[u]:
                if cap>0 and level[v]==-1: level[v]=level[u]+1; q.append(v)
        return level
    def _send_one(self,s,t,level,ptr):
        vertices=[s]; path=[]; bottleneck=[float('inf')]
        while vertices:
            u=vertices[-1]
            if u==t:
                amount=bottleneck[-1]
                for a,i in path:
                    edge=self.adj[a][i]; edge[2]-=amount
                    self.adj[edge[0]][edge[1]][2]+=amount
                return amount
            while ptr[u]<len(self.adj[u]):
                v,rev,cap=self.adj[u][ptr[u]]
                if cap>0 and level[v]==level[u]+1: break
                ptr[u]+=1
            if ptr[u]==len(self.adj[u]):
                vertices.pop(); bottleneck.pop()
                if path: parent,_=path.pop(); ptr[parent]+=1
            else:
                i=ptr[u]; v,rev,cap=self.adj[u][i]
                path.append((u,i)); vertices.append(v); bottleneck.append(min(bottleneck[-1],cap))
        return 0
    def max_flow(self,source,sink):
        if source==sink: raise ValueError('source and sink must differ')
        total=0
        while True:
            level=self._levels(source)
            if level[sink]==-1: return total
            ptr=[0]*len(self.adj)
            while True:
                amount=self._send_one(source,sink,level,ptr)
                if not amount: break
                total+=amount
    def min_cut_reachable(self,source):
        return {i for i,d in enumerate(self._levels(source)) if d!=-1}
    def edges_with_flow(self):
        return [(u,self.adj[u][i][0],cap,cap-self.adj[u][i][2]) for u,i,cap in self.original]

def min_cut_reachable(network,source): return network.min_cut_reachable(source)

def matching_via_flow(left_adj,right_size):
    left_size=len(left_adj); source=left_size+right_size; sink=source+1; net=Dinic(sink+1)
    for u,a in enumerate(left_adj):
        net.add_edge(source,u,1)
        for v in set(a): net.add_edge(u,left_size+v,1)
    for v in range(right_size): net.add_edge(left_size+v,sink,1)
    return net.max_flow(source,sink)

## 正确性、前提与复杂度

每次增广保持容量可行与守恒，反向边保持可撤销性。残量图中源无法到达汇时，从可达集到不可达集的正向边已饱和、反向净流为0，割容量等于流量；任何流都不超过任何割，因此同时得到最大流和最小割证明。

**代价：** 一般图Dinic的标准上界O(V²E)，辅助O(V+E)；本实现阻塞流采用逐路径发送和当前弧。容量使用非负整数，重复max_flow返回在当前残量网络上新增加的流量。支持平行边和自环。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

net=Dinic(4)
for edge in [(0,1,3),(0,2,2),(1,2,1),(1,3,2),(2,3,3)]: net.add_edge(*edge)
print('最大流',net.max_flow(0,3),'源侧最小割',net.min_cut_reachable(0))
show_table(['起点','终点','容量','流量'],net.edges_with_flow())

最大流 5 源侧最小割 {0}


起点,终点,容量,流量
0,1,3,3
0,2,2,2
1,2,1,1
1,3,2,2
2,3,3,3


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
from random import Random
rng=Random(92)
for n in range(2,7):
    for _ in range(25):
        edges=[(rng.randrange(n),rng.randrange(n),rng.randrange(5)) for _ in range(12)]
        net=Dinic(n)
        for e in edges: net.add_edge(*e)
        value=net.max_flow(0,n-1)
        cuts=[]
        for mask in range(1<<n):
            if not mask&1 or mask>>(n-1)&1: continue
            cuts.append(sum(c for u,v,c in edges if mask>>u&1 and not mask>>v&1))
        assert value==min(cuts)
        balance=[0]*n
        for u,v,cap,f in net.edges_with_flow():
            assert 0<=f<=cap; balance[u]-=f; balance[v]+=f
        assert balance[0]==-value and balance[-1]==value and all(x==0 for x in balance[1:-1])
        reach=net.min_cut_reachable(0)
        assert sum(c for u,v,c in edges if u in reach and v not in reach)==value
        assert net.max_flow(0,n-1)==0 # 第二次调用返回新增流量
assert matching_via_flow([[0,1],[0]],2)==2
assert matching_via_flow([[],[]],3)==0
print('N092: 所有本章断言通过')

N092: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 把二分匹配归约为单位容量流。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明本章为通用算法拓展而非1349必需解法。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1349. Maximum Students Taking Exam（extension）：[官方入口](https://leetcode.com/problems/maximum-students-taking-exam/)

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。